## Startup Growth Experiment

 You are analyzing a startup called **Launchly**, a SaaS company that sells productivity software to small businesses.

 The founders want to understand whether **marketing spend actually causes revenue growth**.

 For each startup-month, define:

 - $X$ = marketing spend, measured in thousands of dollars
- $M$ = number of qualified leads generated
- $Y$ = monthly revenue, measured in thousands of dollars

 The startup's underlying data-generating process is:

 $$
X = U_X
$$

 $$
M = 2X + U_M
$$

 $$
Y = 4M + 3X + U_Y
$$

 where

 $$
U_X,U_M,U_Y\sim N(0,1)
$$

 and the three noise variables are independent.

 The causal structure is:

```
              +--------+
              |        ↓
Marketing X → Leads M → Revenue Y
     │                   ↑
     └───────────────────┘
```

 ### Part 1 — Generate the startup data

 Using **NumPy only**, generate data for:

 $$
n=10,000
$$

 startup-month observations.

 Generate $U_X,U_M,U_Y$ from standard normal distributions and use the structural equations above to generate $X,M,Y$.

---

 ### Part 2 — Find the observational relationship

 Calculate the correlation between:

 $$
M \quad\text{and}\quad Y
$$

 Then estimate the regression coefficient:

 $$
\hat{\beta}
=
\frac{\sum_i(M_i-\bar M)(Y_i-\bar Y)}
{\sum_i(M_i-\bar M)^2}
$$

 Interpret the coefficient in startup terms.

 **Question:** Does this coefficient represent the causal effect of generating one additional unit of $M$ on revenue?

---

 ### Part 3 — Perform an intervention

 The founders ask:

 > "What would happen to revenue if we could force every startup-month to generate exactly 10 qualified leads?"

 Represent this as:

 $$
do(M=10)
$$

 Using the structural equation for revenue,

 $$
Y=4M+3X+U_Y
$$

 calculate:

 $$
Y_{do(M=10)}
$$

 for every observation.

 Then calculate the average revenue under this intervention:

 $$
E[Y\mid do(M=10)]
$$

---

 ### Part 4 — Calculate the causal effect

 What is the causal effect of increasing qualified leads from $M=5$ to $M=10$?

 Calculate:

 $$
E[Y\mid do(M=10)]
-
E[Y\mid do(M=5)]
$$

 What does this number mean for Launchly?

---

 ### Part 5 — Counterfactual startup

 Now focus on **one particular startup-month**, observation $i=0$.

 Suppose its factual values are:

 $$
M_0 = ?
$$

 $$
Y_0 = ?
$$

 The founders ask:

 > "For this exact startup-month, what would revenue have been if it had generated 10 qualified leads instead?"

 This is the counterfactual:

 $$
Y_0(M=10)
$$

 Using the original $U_{Y,0}$, calculate:

 $$
Y_0(10)=4(10)+3X_0+U_{Y,0}
$$

 **Important:** Do not generate a new $U_Y$. You are asking about the **same startup-month in a different hypothetical situation**.

---

 ### Part 6 — Compare factual and counterfactual

 Calculate:

 $$
Y_0(M_0)
$$

 and

 $$
Y_0(10)
$$

 Then calculate the individual counterfactual effect:

 $$
Y_0(10)-Y_0(M_0)
$$


---

 ### Final conceptual question


 $$
P(Y\mid M)
$$

 $$
P(Y\mid do(M=m))
$$

 and

 $$
Y_i(m)
$$


1. **What we observe**
2. **What happens when we intervene**
3. **What would have happened to this particular startup-month**

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

In [6]:
np.random.seed(42)
n=10000

Ux = np.random.normal(0,1,n) #spend
Um = np.random.normal(0,1,n) #qualified leads
Uy = np.random.normal(0,1,n)#revenue

X = Ux
M = 2*X + Um
Y = 4*M + 3*X + Uy

In [8]:
# find the observational relationship
# find correlation between M (leads) and Y (revenue)
corr = np.corrcoef(M, Y)[0, 1]

beta_hat = np.sum((M  - M.mean()) * (Y - Y.mean())) / \
        np.sum((M - M.mean()) ** 2)

beta_hat

np.float64(5.220247380957723)

In [10]:
# Part 3: Intervention do(M=10)
Y_do_10 = 4 * 10 + 3 * X + Uy
E_Y_do_10 = np.mean(Y_do_10)

# Part 4: Intervention do(M=5) and the causal effect
Y_do_5 = 4 * 5 + 3 * X + Uy
E_Y_do_5 = np.mean(Y_do_5)

causal_effect = E_Y_do_10 - E_Y_do_5
print(causal_effect)

20.000000000000004


In [13]:
# Part 5: Counterfactual for observation i = 0
i = 0
X_0 = X[i]
UY_0 = Uy[i]

# What would revenue have been if M had been 10 instead?
Y_0_counterfactual = 4 * 10 + 3 * X_0 + UY_0
Y_0_counterfactual

np.float64(41.83842870670052)

**Observational ($P(Y \mid M)$)** represents what we see in the data. Because marketing spend ($X$) causes both leads ($M$) and revenue ($Y$), looking at the observational relationship captures both the direct effect of leads *and* the hidden confounding effect of marketing spend, which is why the observational regression coefficient ($\hat{\beta} \approx 5.2$) is higher than the direct causal coefficient ($4$).

**Interventional ($P(Y \mid do(M=m))$)** represents what happens on average if we systematically force every startup-month to a specific lead count (e.g., $M=10$). By actively intervening, we sever the incoming arrow from marketing spend to leads, isolating the true, unconfounded causal effect of leads on revenue.

**Counterfactual ($Y_i(m)$)** asks about a specific, individual startup-month ($i=0$) after the fact. It answers what *would have* happened to that exact company month if we had changed $M$ while holding its unique, unobserved background noise ($U_{Y,0}$) completely constant.